# schema_finder_v2
fdriver 0단계 - investar DB 테이블 인벤토리 수집 (읽기 전용)

- 조회 대상: information_schema + 표준 규격(STD-1) 테이블의 기간·건수 (SELECT만 수행, 쓰기 작업 없음)
- v2 추가: `period_end` 컬럼이 있는 테이블의 실제 기간(MIN/MAX)과 행 수를 `investar_coverage_*.csv`로 저장
- 결과 저장: `C:\Users\82108\OneDrive\INVESTMENT\한국주식\miscell`

## 1. 설정

In [1]:
import os
import sys
import glob
from datetime import datetime

import pandas as pd

# 결과 저장 폴더
OUT_DIR = r"C:\Users\82108\OneDrive\INVESTMENT\한국주식\miscell"

# config.py가 있는 폴더 (investment_strategy\DATA)
# 아래 경로가 다르면 실제 경로로 수정. 비워두면(None) 자동 탐색 시도
CONFIG_DIR = None

SCHEMA = "investar"
STAMP = datetime.now().strftime("%Y%m%d_%H%M")

os.makedirs(OUT_DIR, exist_ok=True)
print("결과 저장 폴더:", OUT_DIR)

결과 저장 폴더: C:\Users\82108\OneDrive\INVESTMENT\한국주식\miscell


## 2. config.py 불러오기

In [2]:
def find_config_dir():
    """config.py 위치 탐색: 현재 폴더 -> OneDrive INVESTMENT 하위의 investment_strategy\\DATA"""
    if os.path.exists(os.path.join(os.getcwd(), "config.py")):
        return os.getcwd()
    roots = [
        r"C:\Users\82108\OneDrive\INVESTMENT",
        r"C:\Users\82108\OneDrive",
        r"C:\Users\82108",
    ]
    for root in roots:
        if not os.path.isdir(root):
            continue
        for depth in range(1, 5):
            pattern = os.path.join(root, *(["*"] * (depth - 1)), "investment_strategy", "DATA", "config.py")
            hits = glob.glob(pattern)
            if hits:
                return os.path.dirname(hits[0])
    return None

if CONFIG_DIR is None:
    CONFIG_DIR = find_config_dir()

if CONFIG_DIR is None:
    raise FileNotFoundError(
        "config.py를 찾지 못했습니다. 1번 셀의 CONFIG_DIR에 config.py가 있는 폴더 경로를 직접 입력하세요."
    )

if CONFIG_DIR not in sys.path:
    sys.path.append(CONFIG_DIR)

import config
engine = config.get_engine(config.get_db_info())
print("config 위치:", CONFIG_DIR)
print("DB 엔진 생성 완료")

config 위치: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
DB 엔진 생성 완료


## 3. investar 테이블 목록 조회

In [3]:
sql_tables = f"""
SELECT TABLE_NAME,
       TABLE_TYPE,
       ENGINE,
       TABLE_ROWS,
       DATA_LENGTH,
       INDEX_LENGTH,
       CREATE_TIME,
       UPDATE_TIME,
       TABLE_COLLATION
FROM information_schema.TABLES
WHERE TABLE_SCHEMA = '{SCHEMA}'
ORDER BY TABLE_NAME
"""

df_tables = pd.read_sql(sql_tables, engine)

# 보기 편하도록 용량(MB) 컬럼 추가
df_tables["DATA_MB"] = (df_tables["DATA_LENGTH"] / 1024**2).round(1)
df_tables["INDEX_MB"] = (df_tables["INDEX_LENGTH"] / 1024**2).round(1)

print("테이블 수:", len(df_tables))
print("총 데이터 용량(MB):", round(df_tables["DATA_MB"].sum(), 1))

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)
df_tables[["TABLE_NAME", "TABLE_TYPE", "TABLE_ROWS", "DATA_MB", "UPDATE_TIME", "TABLE_COLLATION"]]

테이블 수: 131
총 데이터 용량(MB): 28581.7


,TABLE_NAME,TABLE_TYPE,TABLE_ROWS,DATA_MB,UPDATE_TIME,TABLE_COLLATION
0,amazon_asin_master,BASE TABLE,50.0,0.0,NaT,utf8mb4_general_ci
1,amazon_brand_index,BASE TABLE,3289.0,0.3,NaT,utf8mb4_general_ci
2,amazon_bsr_daily,BASE TABLE,32320.0,2.5,NaT,utf8mb4_general_ci
3,amazon_collect_log,BASE TABLE,6.0,0.0,NaT,utf8mb4_general_ci
4,apr_revenue_quarterly,BASE TABLE,27.0,0.0,NaT,utf8mb4_general_ci
5,apr_us_google_trends,BASE TABLE,1310.0,0.2,NaT,utf8mb4_general_ci
6,apr_us_quarterly_actual,BASE TABLE,0.0,0.0,NaT,utf8mb4_general_ci
7,apr_us_trends_metrics,BASE TABLE,262.0,0.1,NaT,utf8mb4_general_ci
8,bok_economic_indicators,BASE TABLE,12375.0,2.5,NaT,utf16_general_ci
9,correlation_price_hscode,BASE TABLE,4483063.0,312.8,NaT,utf16_general_ci


## 4. 컬럼 구조 조회 (매핑 확정용)

In [4]:
sql_cols = f"""
SELECT TABLE_NAME,
       ORDINAL_POSITION,
       COLUMN_NAME,
       COLUMN_TYPE,
       IS_NULLABLE,
       COLUMN_KEY,
       COLLATION_NAME
FROM information_schema.COLUMNS
WHERE TABLE_SCHEMA = '{SCHEMA}'
ORDER BY TABLE_NAME, ORDINAL_POSITION
"""

df_cols = pd.read_sql(sql_cols, engine)
print("컬럼 수:", len(df_cols))
df_cols.head(30)

컬럼 수: 1889


,TABLE_NAME,ORDINAL_POSITION,COLUMN_NAME,COLUMN_TYPE,IS_NULLABLE,COLUMN_KEY,COLLATION_NAME
0,amazon_asin_master,1,asin,varchar(16),NO,PRI,utf8mb4_general_ci
1,amazon_asin_master,2,domain,varchar(6),NO,PRI,utf8mb4_general_ci
2,amazon_asin_master,3,ticker,varchar(20),YES,MUL,utf8mb4_general_ci
3,amazon_asin_master,4,company,varchar(60),YES,,utf8mb4_general_ci
4,amazon_asin_master,5,brand,varchar(80),YES,,utf8mb4_general_ci
5,amazon_asin_master,6,title,varchar(400),YES,,utf8mb4_general_ci
6,amazon_asin_master,7,parent_asin,varchar(16),YES,MUL,utf8mb4_general_ci
7,amazon_asin_master,8,root_cat_id,bigint(20),YES,,None
8,amazon_asin_master,9,root_cat_name,varchar(120),YES,,utf8mb4_general_ci
9,amazon_asin_master,10,is_device,tinyint(4),NO,,None


## 5. investar 외 스키마 확인 (외부 저장소 여부 점검용)

In [5]:
sql_schemas = """
SELECT s.SCHEMA_NAME,
       s.DEFAULT_COLLATION_NAME,
       COUNT(t.TABLE_NAME) AS N_TABLES
FROM information_schema.SCHEMATA s
LEFT JOIN information_schema.TABLES t
       ON t.TABLE_SCHEMA = s.SCHEMA_NAME
WHERE s.SCHEMA_NAME NOT IN ('information_schema', 'mysql', 'performance_schema', 'sys')
GROUP BY s.SCHEMA_NAME, s.DEFAULT_COLLATION_NAME
ORDER BY s.SCHEMA_NAME
"""

df_schemas = pd.read_sql(sql_schemas, engine)
df_schemas

,SCHEMA_NAME,DEFAULT_COLLATION_NAME,N_TABLES
0,investar,utf16_general_ci,131


## 5-1. 표준 규격 테이블 기간·건수 (v2 추가)
`period_end` 컬럼이 있는 테이블만 조회합니다. 기간은 인덱스로 MIN/MAX만 구하고,
500만 행이 넘는 테이블은 정확한 COUNT 대신 추정치를 씁니다 (0단계 방침).


In [6]:
from sqlalchemy import text

BIG_ROWS = 5_000_000
std_tables = sorted(set(df_cols.loc[df_cols["COLUMN_NAME"] == "period_end", "TABLE_NAME"]))
base = df_tables.set_index("TABLE_NAME")

rows = []
with engine.connect() as cx:
    for t in std_tables:
        if base.loc[t, "TABLE_TYPE"] != "BASE TABLE":
            continue
        est = base.loc[t, "TABLE_ROWS"]
        pmin, pmax = cx.execute(text(f"SELECT MIN(period_end), MAX(period_end) FROM `{t}`")).fetchone()
        if est is not None and est == est and est > BIG_ROWS:
            n, how = int(est), "estimate"
        else:
            n, how = cx.execute(text(f"SELECT COUNT(*) FROM `{t}`")).scalar(), "exact"
        cols = set(df_cols.loc[df_cols["TABLE_NAME"] == t, "COLUMN_NAME"])
        key = next((k for k in ("series_id", "ticker", "nat_code", "idx_name") if k in cols), None)
        n_keys = cx.execute(text(f"SELECT COUNT(DISTINCT `{key}`) FROM `{t}`")).scalar() if key else None
        rows.append({"TABLE_NAME": t, "ROWS": n, "ROWS_HOW": how, "PERIOD_MIN": pmin, "PERIOD_MAX": pmax,
                     "KEY_COLUMN": key, "N_KEYS": n_keys,
                     "COLLATION": base.loc[t, "TABLE_COLLATION"]})

df_cov = pd.DataFrame(rows)
print("표준 규격 테이블:", len(df_cov))
df_cov

표준 규격 테이블: 18


,TABLE_NAME,ROWS,ROWS_HOW,PERIOD_MIN,PERIOD_MAX,KEY_COLUMN,N_KEYS,COLLATION
0,kr_airline_traffic,86436,exact,2015-01-31,2026-08-31,series_id,1990.0,utf8mb4_general_ci
1,kr_airline_traffic_vintage,86436,exact,2015-01-31,2026-08-31,series_id,1990.0,utf8mb4_general_ci
2,kr_komis_mineral_price,114509,exact,2000-12-25,2026-09-30,series_id,32.0,utf8mb4_general_ci
3,kr_komis_mineral_price_vintage,114510,exact,2000-12-25,2026-09-30,series_id,32.0,utf8mb4_general_ci
4,kr_kosis_data,43252,exact,2010-01-31,2026-08-31,series_id,336.0,utf8mb4_general_ci
5,kr_kosis_data_vintage,43252,exact,2010-01-31,2026-08-31,series_id,336.0,utf8mb4_general_ci
6,kr_stock_credit_balance_daily,1194,exact,2021-11-09,2026-09-29,None,NaN,utf8mb4_general_ci
7,kr_stock_credit_balance_daily_vintage,1194,exact,2021-11-09,2026-09-29,None,NaN,utf8mb4_general_ci
8,kr_stock_index_daily,0,exact,None,None,idx_name,0.0,utf8mb4_general_ci
9,kr_stock_index_daily_vintage,0,exact,None,None,idx_name,0.0,utf8mb4_general_ci


## 6. 결과 저장

In [7]:
path_tables  = os.path.join(OUT_DIR, f"investar_tables_{STAMP}.csv")
path_cols    = os.path.join(OUT_DIR, f"investar_columns_{STAMP}.csv")
path_schemas = os.path.join(OUT_DIR, f"db_schemas_{STAMP}.csv")
path_cov     = os.path.join(OUT_DIR, f"investar_coverage_{STAMP}.csv")

df_tables.to_csv(path_tables, index=False, encoding="utf-8-sig")
df_cols.to_csv(path_cols, index=False, encoding="utf-8-sig")
df_schemas.to_csv(path_schemas, index=False, encoding="utf-8-sig")
df_cov.to_csv(path_cov, index=False, encoding="utf-8-sig")

print("저장 완료")
print(" -", path_tables)
print(" -", path_cols)
print(" -", path_schemas)
print(" -", path_cov)

저장 완료
 - C:\Users\82108\OneDrive\INVESTMENT\한국주식\miscell\investar_tables_20261001_2049.csv
 - C:\Users\82108\OneDrive\INVESTMENT\한국주식\miscell\investar_columns_20261001_2049.csv
 - C:\Users\82108\OneDrive\INVESTMENT\한국주식\miscell\db_schemas_20261001_2049.csv
 - C:\Users\82108\OneDrive\INVESTMENT\한국주식\miscell\investar_coverage_20261001_2049.csv


## 참고
- `TABLE_ROWS`는 InnoDB에서 **추정치**입니다. 정확한 행 수는 다음 단계 인벤토리에서 별도로 측정합니다.
- `UPDATE_TIME`은 InnoDB 테이블에서 비어 있을(NULL) 수 있습니다. 최신 적재일은 다음 단계에서 날짜 컬럼 MAX로 확인합니다.